# Intro to GraphRAG with Google Cloud Spanner Graph

Spanner Graph now [integrates seamlessly with LangChain](https://cloud.google.com/python/docs/reference/langchain-google-spanner/latest#spanner-graph-store-usage), making it easier to build GraphRAG applications.

Instead of simply retrieving relevant text snippets based on keyword similarity, GraphRAG takes a more sophisticated, structured approach to Retrieval Augmented Generation. It involves creating a knowledge graph from the text, organizing it hierarchically, summarizing key concepts, and then using this structured information to enhance the accuracy and depth of responses.


### Objectives

In this tutorial, you will see a complete walkthrough of building a question-answering system using the GraphRAG method. You'll learn how to create a knowledge graph from scratch, store it efficiently in Spanner Graph, a functional FAQ system with Langchain agent.

Google Cloud [Spanner](https://cloud.google.com/spanner) is a highly scalable database that combines unlimited scalability with relational semantics, such as secondary indexes, strong consistency, schemas, and SQL providing 99.999% availability in one easy solution.

This notebook goes over how to use `Spanner Graph` for GraphRAG with the custom retriever `SpannerGraphVectorContextRetriever` and compares the response of GraphRAG with conventional RAG.


In this lab you will:
 * Learn about knowledge graph extraction by leveraging structured outputs and prompts for entity and relation recognition.
 * Use structured outputs to process and store extracted entities in a graph database.
 * Build a simple Graph Retrieval-Augmented Generation (Graph RAG) system based on the extracted knowledge graph.

The following example use in this notebook is based on a modified and extended version of the sample code from public [Spanner for LangChain](https://github.com/googleapis/langchain-google-spanner-python#spanner-for-langchain) repository:



## Prepare Environment
### Import required dependencies

In [ ]:
import copy
import json
import os
import pprint
import textwrap
import uuid
import warnings
from google.cloud import spanner
from google.cloud.spanner_admin_database_v1.types import spanner_database_admin
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_spanner import (
    SpannerGraphStore,
    SpannerGraphVectorContextRetriever,
    SpannerVectorStore,
)
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.api_core.exceptions import NotFound
from typing import List, Optional, Union
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore", category=DeprecationWarning)

from IPython.display import Markdown

### Specify the embedding and generative language models

In [ ]:
EMBEDDING_MODEL = "text-embedding-004"
GENERATIVE_MODEL = "gemini-2.5-flash"

### Set Google Cloud project ID environment variable

In [ ]:
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"
%env GOOGLE_CLOUD_PROJECT={PROJECT_ID}

### Enable Spanner API
The `langchain-google-spanner` package requires that you [enable the Spanner API](https://console.cloud.google.com/flows/enableapi?apiid=spanner.googleapis.com) in your Google Cloud Project.

In [ ]:
!gcloud services enable spanner.googleapis.com

## Setup Spanner Graph Database

### Set Spanner database values
Find your database values, in the [Spanner Instances page](https://console.cloud.google.com/spanner?_ga=2.223735448.2062268965.1707700487-2088871159.1707257687).

In [ ]:
INSTANCE = "graphrag-instance-v1"
DATABASE = "graphrag"
GRAPH_NAME = "retail_demo_graph_v1"

### Create a Cloud Spanner instance
To create a Cloud Spanner instance using gcloud, use the gcloud spanner instances create command, specifying a unique instance ID, configuration (region/multi-region), description, and compute capacity (nodes or processing units).

In [ ]:
!gcloud spanner instances describe {INSTANCE} > /dev/null 2>&1 || gcloud spanner instances create {INSTANCE} --config=regional-us-central1 --description="Graph RAG Instance" --processing-units=100 --edition=ENTERPRISE

### Define helper method to create a Spanner database
Helper method to create a spanner database and table to store the graph with nodes and edges created in graph:

In [ ]:
def create_database(project_id, instance_id, database_id):
    """Creates a database and tables for sample data."""

    spanner_client = spanner.Client(project=project_id)
    database_admin_api = spanner_client.database_admin_api
    
    # Construct the full path to the database
    db_path = database_admin_api.database_path(
        spanner_client.project, instance_id, database_id
    )

    # 1. Attempt to get the database
    try:
        database_admin_api.get_database(name=db_path)
        print(f"Database '{database_id}' already exists. Skipping creation.")
        return
    except NotFound:
        # 2. Catch the NotFound exception and proceed with creation
        print(f"Database '{database_id}' not found. Creating...")

    request = spanner_database_admin.CreateDatabaseRequest(
        parent=database_admin_api.instance_path(
            spanner_client.project, instance_id
        ),
        create_statement=f"CREATE DATABASE `{database_id}`",
        extra_statements=[],
    )

    operation = database_admin_api.create_database(request=request)

    print("Waiting for operation to complete...")
    OPERATION_TIMEOUT_SECONDS = 60
    database = operation.result(OPERATION_TIMEOUT_SECONDS)

    print(
        "Created database {} on instance {}".format(
            database.name,
            database_admin_api.instance_path(
                spanner_client.project, instance_id
            ),
        )
    )

### Creating a Spanner database
Creating a Spanner database using helper method:

In [ ]:
create_database(PROJECT_ID, INSTANCE, DATABASE)

### Initialize SpannerGraphStore

To initialize the `SpannerGraphStore` class you need to provide 3 required arguments and other arguments are optional and only need to pass if it's different from default ones

1.   a Spanner instance id;
2.   a Spanner database id belongs to the above instance id;
3.   a Spanner graph name used to create a graph in the above database.

In [ ]:
graph_store = SpannerGraphStore(
    instance_id=INSTANCE,
    database_id=DATABASE,
    graph_name=GRAPH_NAME,
)

## Create Knowledge Graph

### Downloading unstructured text documents
Downloading simple unstructured text documents for the next processing steps.

In [ ]:
# Deletes existing retaildata.zip if present
![ -f retaildata.zip ] && rm retaildata.zip && echo "Existing file deleted."
# Download original retaildata.zip if present
!wget https://raw.githubusercontent.com/googleapis/langchain-google-spanner-python/main/samples/retaildata.zip
# Deletes existing the folder to avoid duplicates
!rm -rf retaildata/
# unpack the retaildata.zip arcive
!unzip -o "retaildata.zip"

### Loading text files
Iterate through all subdirectories within the "retaildata/" folder and use TextLoader
with DirectoryLoader to load all text files from each of those subfolders into a collections.

In [ ]:
path = "retaildata/"
directories = [
    item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))
]

document_lists = []
for directory in directories:
    loader = DirectoryLoader(
        os.path.join(path, directory), glob="**/*.txt", loader_cls=TextLoader
    )
    document_lists.append(loader.load())

### Define Pydantic Extraction Schemas for Structured Output

To reliably extract a knowledge graph from unstructured text, we need to guarantee that our Large Language Model (LLM) returns data in a predictable, structured format. In this step, we use **Pydantic** to define a strict data schema.

By defining these base models, we create a blueprint that instructs the LLM exactly how to structure the entities (nodes) and their connections (relationships):

* **`PropertyExtraction`**: Captures granular key-value pairs for specific attributes (e.g., `{"key": "location", "value": "Paris"}`).
* **`NodeExtraction`**: Represents an entity (like a Person or Organization). It includes a unique ID, its type, and a list of its properties.
* **`RelExtraction`**: Represents the directed connection between two nodes. It maps the source node to the target node, defines the relationship type (enforced in UPPERCASE), and lists any relationship-specific properties.
* **`GraphExtraction`**: The master output wrapper containing the complete list of all extracted nodes and relationships.


In [ ]:
class PropertyExtraction(BaseModel):
    key: str = Field(description="The name of the property.")
    value: str = Field(description="The value of the property.")

class NodeExtraction(BaseModel):
    id: str = Field(description="The unique identifier or name of the entity.")
    type: str = Field(description="The type or label of the entity.")
    properties: List[PropertyExtraction] = Field(default_factory=list, description="List of attributes for this entity.")

class RelExtraction(BaseModel):
    source_id: str = Field(description="The ID of the source node.")
    source_type: str = Field(description="The type or label of the source node.")
    target_id: str = Field(description="The ID of the target node.")
    target_type: str = Field(description="The type or label of the target node.")
    type: str = Field(description="The type of relationship connecting them. Use UPPERCASE.")
    properties: List[PropertyExtraction] = Field(default_factory=list, description="List of attributes for this relationship.")

class GraphExtraction(BaseModel):
    nodes: List[NodeExtraction] = Field(description="List of nodes extracted from text.")
    relationships: List[RelExtraction] = Field(description="List of relationships extracted from text.")

### Standardize Property Keys

When extracting entities and their attributes from unstructured text, the resulting property keys can often be inconsistent. 
For example, an extraction model might return `deal end date`, `deal_end_date`, or `Deal End Date` across different runs. 

To maintain a clean, predictable schema in our knowledge graph or database, we must normalize these strings. Run the cell below to define a helper function that automatically formats any incoming property keys into standard `camelCase`.

In [ ]:
# --- Helper function ---
def format_property_key(s: str) -> str:
    """CamelCases property keys (e.g., 'deal end date' -> 'dealEndDate')"""
    words = s.replace("_", " ").split()
    if not words:
        return s
    first_word = words[0].lower()
    capitalized_words = [word.capitalize() for word in words[1:]]
    return "".join([first_word] + capitalized_words)

### Create Custom Graph Extractor
The CustomGraphExtractor class designed to use a Large Language Model (LLM) to automatically extract structured Knowledge Graph - consisting of nodes (entities) and relationships—from unstructured text documents.

Its primary purpose is to enforce strict schema constraints by injecting user-defined allowed node types, relationship categories, and property limits directly into the LLM's prompt template. After the LLM generates the data, the code cleans and processes the output by standardizing text casing, and safely deduplicating overlapping entities and edges to return well-formed GraphDocument objects suitable for a graph database.

In [ ]:
from string import Template
class CustomGraphExtractor:
    def __init__(
        self, 
        llm, 
        allowed_nodes: Optional[List[str]] = None, 
        allowed_relationships: Optional[List[str]] = None,
        node_properties: Union[bool, List[str]] = False,
        strict_mode: bool = True
    ):
        # using llm client with structured output
        self.structured_llm = llm.with_structured_output(GraphExtraction)
        
        # Lowercase allowed lists for case-insensitive checking later
        self.allowed_nodes = [n.lower() for n in allowed_nodes] if allowed_nodes else None
        self.allowed_relationships = [r.lower() for r in allowed_relationships] if allowed_relationships else None
        
        self.node_properties = node_properties
        self.strict_mode = strict_mode

        # Build prompt constraints
        self.node_constraints = ""
        if self.allowed_nodes:
            self.node_constraints = f"\n- Node Types: You MUST strictly use ONLY these types: {', '.join(self.allowed_nodes)}."
        print("="*80)
        print(f"Node Types Constraints: {self.node_constraints}")
        print("="*80)
            
        rel_constraints = ""
        if self.allowed_relationships:
            self.rel_constraints = f"\n- Relationship Types: You MUST strictly use ONLY these types: {', '.join(self.allowed_relationships)}."

        print(f"Relations Constraints: {self.allowed_relationships}")
        print("="*80)
            
        self.prop_constraints = ""
        if isinstance(self.node_properties, list):
            self.prop_constraints = f"\n- Node Properties: You are strictly limited to these property keys: {', '.join(self.node_properties)}. Map them as key/value pairs."
        elif self.node_properties is False:
            self.prop_constraints = "\n- Node Properties: Do not extract any properties."

        print(f"Node Properties Constraints: {self.prop_constraints}")
        print("="*80)

        self.llm_prompt_template = Template(
            f"""
            Extract all relevant entities and relationships from the text below according to these strict structural rules:
            {self.node_constraints}
            {self.rel_constraints}
            {self.prop_constraints}
            
            Text to extract from:
            $page_content
            """
        )


    def convert_to_graph_documents(self, documents: List[Document]) -> List[GraphDocument]:
        
        graph_docs = []
        
        for doc in documents:

            # --- 1. Prepare final prompt by addig actual input text document:
            prompt = self.llm_prompt_template.substitute(page_content=doc.page_content)

            # --- 2. Using prompt to call llm with structured output:
            extracted_data: GraphExtraction = self.structured_llm.invoke(prompt)
            
            # --- 3. Map Nodes (With Proper Deduplication, Merging & Auto-Naming) ---
            
            nodes_dict = {}
            for n in extracted_data.nodes:
                n_id = n.id.title() if isinstance(n.id, str) else n.id
                n_type = n.type.capitalize() if n.type else "Node"
                
                # Strict mode check
                if self.strict_mode and self.allowed_nodes and n_type.lower() not in self.allowed_nodes:
                    continue
                
                # Format properties
                raw_props = {format_property_key(prop.key): prop.value for prop in n.properties}
                filtered_props = {}
                if isinstance(self.node_properties, list):
                    allowed_props_formatted = [format_property_key(p) for p in self.node_properties]
                    filtered_props = {k: v for k, v in raw_props.items() if k in allowed_props_formatted}
                elif self.node_properties is True:
                    filtered_props = raw_props
                
                # Ensure every explicit node has a 'name' property
                if "name" not in filtered_props:
                    filtered_props["name"] = n.id # Preserve exact original casing for the name
                
                # Use Composite Key for deduplication
                node_key = (str(n_id).lower(), n_type.lower())
                
                if node_key in nodes_dict:
                    # Merge properties carefully to avoid Pydantic immutable errors
                    existing_node = nodes_dict[node_key]
                    merged_props = existing_node.properties.copy() if existing_node.properties else {}
                    merged_props.update(filtered_props)
                    
                    # Re-instantiate the node with merged properties
                    nodes_dict[node_key] = Node(id=existing_node.id, type=existing_node.type, properties=merged_props)
                else:
                    nodes_dict[node_key] = Node(id=n_id, type=n_type, properties=filtered_props)
            
            # --- 4. Map Relationships (With Strict Checking, Deduplication & Fallback Naming) ---
            relationships = []
            seen_relationships = set()
            for r in extracted_data.relationships:
                r_type = r.type.replace(" ", "_").upper()
                
                # Strict check on relationship type
                if self.strict_mode and self.allowed_relationships and r_type.lower() not in self.allowed_relationships:
                    continue
                
                s_id = r.source_id.title() if isinstance(r.source_id, str) else r.source_id
                s_type = r.source_type.capitalize() if r.source_type else "Node"
                t_id = r.target_id.title() if isinstance(r.target_id, str) else r.target_id
                t_type = r.target_type.capitalize() if r.target_type else "Node"
                
                # Strict check on connecting node types
                if self.strict_mode and self.allowed_nodes:
                    if s_type.lower() not in self.allowed_nodes or t_type.lower() not in self.allowed_nodes:
                        continue
                
                # Construct composite keys for source and target
                source_key = (str(s_id).lower(), s_type.lower())
                target_key = (str(t_id).lower(), t_type.lower())
                
                # Ensure nodes exist in nodes_dict (Blind Fallback with naming)
                source_node = nodes_dict.get(source_key)
                if not source_node:
                    source_node = Node(id=s_id, type=s_type, properties={"name": r.source_id})
                    nodes_dict[source_key] = source_node
                    
                target_node = nodes_dict.get(target_key)
                if not target_node:
                    target_node = Node(id=t_id, type=t_type, properties={"name": r.target_id})
                    nodes_dict[target_key] = target_node
                
                # --- Edge Deduplication Signature ---
                # A relationship is unique based on Source, Target, and Relationship Type
                rel_signature = (source_key, target_key, r_type)
                
                if rel_signature not in seen_relationships:
                    seen_relationships.add(rel_signature)
                    rel_props = {format_property_key(prop.key): prop.value for prop in r.properties}
                    
                    relationships.append(
                        Relationship(
                            source=source_node,
                            target=target_node,
                            type=r_type,
                            properties=rel_props
                        )
                    )
                    
            graph_docs.append(
                GraphDocument(
                    nodes=list(nodes_dict.values()),
                    relationships=relationships,
                    source=doc
                )
            )
            
        return graph_docs

### Initialize CustomGraphExtractor
Creating instance of the CustomGraphExtractor to process unstructured text.
Configuring the extractor with a strict schema—defining allowable product types, pricing properties, and sales relationships—ensuring the resulting knowledge graph.

In [ ]:
llm = ChatVertexAI(model=GENERATIVE_MODEL, temperature=0)
llm_graph_extractor = CustomGraphExtractor(
    llm=llm,
    allowed_nodes=["Category", "Segment", "Tag", "Product", "Bundle", "Deal"],
    allowed_relationships=[
        "In_Category",
        "Tagged_With",
        "In_Segment",
        "In_Bundle",
        "Is_Accessory_Of",
        "Is_Upgrade_Of",
        "Has_Deal",
    ],
    node_properties=[
        "name",
        "price",
        "weight",
        "deal_end_date",
        "features",
    ],
)

### Define utility method to print extracted graph
Iterate through the output to neatly print the extracted nodes, properties, and relationships, allowing you to verify that the strict schema was applied correctly.


In [ ]:
def print_graph(graph_documents):
    for graph_document in graph_documents: 
        print("=========================================")
        print(f"RAW TEXT:{graph_document.source.page_content[:100]} ...")
        print("=========================================")
        print(f"NODES (Total: {len(graph_document.nodes)})")
        print("=========================================")
        # Iterate over all nodes in the graph document
        for node in graph_document.nodes:
            print(f"Node ID: {node.id}")
            print(f"  Type: {node.type}")
            print(f"  Properties:")
            for node_prop in node.properties:
                if node_prop == "embedding":
                    print(f"\t{node_prop} = {node.properties[node_prop][:3]} ...")
                else:
                    print(f"\t{node_prop} = {node.properties[node_prop]}")
            print("-" * 40)
            
        print("=========================================")
        print(f"RELATIONSHIPS (Total: {len(graph_document.relationships)})")
        print("=========================================")
        # Iterate over all relationships in the graph document
        for rel in graph_document.relationships:
            # Relationships contain full Node objects for source and target
            print(f"[{rel.source.id}] --({rel.type})--> [{rel.target.id}]")
            if rel.properties:
                print(f"  Relationship Properties: {rel.properties}")

### Single text document test
Run a sample text ddocument containing product specifications through the CustomGraphExtractor to extract Knowledge Graph. 

In [ ]:
# 1. Document raw text
text_string="""\n DataSafe Vault\n\nPrice: $39.99\n
Manufacturer: SecureTech Solutions\n\nDescription: Keep your sensitive data safe and secure with the DataSafe Vault.
This rugged and encrypted hard drive case provides superior protection against
unauthorized access, accidental damage, and environmental hazards. Perfect for safeguarding personal files, business documents,
or any data you need to keep private.\n\nFeatures:\n\n*   Military-grade encryption: 256-bit AES hardware encryption ensures
your data is unreadable without the correct password.\n*   Durable construction: Constructed with a shock-resistant polymer
shell and a water-resistant seal to withstand harsh conditions.\n*   Tamper-proof design: Features a tamper-evident seal to 
alert you to any unauthorized access attempts.\n*   Password protection: Customizable password with alphanumeric and special 
character support.\n*   LED indicator: Clearly displays drive activity and encryption status.
*   USB 3.0 connectivity: High-speed data transfer for quick and efficient backups.\n\nSpecifications:\n
*   Compatible drive size: 2.5-inch SATA HDD/SSD\n*   Interface: USB 3.0 (backward compatible with USB 2.0)
*   Encryption type: 256-bit AES hardware encryption\n*   Material: High-impact polymer, rubber gasket
*   Dimensions: 5.0 x 3.2 x 0.75 inches\n*   Weight: 4 ounces
*   Operating temperature: 32°F to 140°F (0°C to 60°C)
*   Power supply: USB powered\n*   Accessory of: TitanDrive X5000
Tags: ['Storage', 'Data']\n
"""

# 2. Create the Document
test_doc = Document(page_content=text_string)
test_extracted_graph = llm_graph_extractor.convert_to_graph_documents([test_doc])
print_graph(test_extracted_graph)

### Single text document test
Process all text documents through the custom LLM extractor to generate a comprehensive collection of structured knowledge graph documents. It then enriches these graphs by generating Vertex AI vector embeddings for the product "features" properties, enabling similarity-based semantic search capabilities before printing the final structure.

In [ ]:
graph_documents = []
for document_list in document_lists:
    graph_documents.extend(
        llm_graph_extractor.convert_to_graph_documents(document_list)
    )

#Add embeddings to the graph documents for Product nodes
embedding_service = VertexAIEmbeddings(model_name=EMBEDDING_MODEL)
for graph_document in graph_documents:
    for node in graph_document.nodes:
        if "features" in node.properties:
            node.properties["embedding"] = embedding_service.embed_query(
                node.properties["features"]
            )

print_graph(graph_documents[:10])

#### Post process extracted nodes and edges
Apply your domain knowledge to clean up and make desired fixes to the
generated graph in the earlier step.

In [ ]:
# set of all valid products
products = set()

def prune_invalid_products():
    for graph_document in graph_documents:
        nodes_to_remove = []
        relationships_to_remove = []
        for node in graph_document.nodes:
            if node.type == "Product" and "features" not in node.properties:
                nodes_to_remove.append(node)
            else:
                products.add(node.id)
        for node in nodes_to_remove:
            graph_document.nodes.remove(node)


def prune_invalid_segments(valid_segments):
    for graph_document in graph_documents:
        nodes_to_remove = []
        for node in graph_document.nodes:
            if node.type == "Segment" and node.id not in valid_segments:
                nodes_to_remove.append(node)
        for node in nodes_to_remove:
            graph_document.nodes.remove(node)


def is_not_a_listed_product(node):
    if node.type == "Product" and node.id not in products:
        return True
    return False


def fix_directions(relation_name, wrong_source_type):
    for graph_document in graph_documents:
        for relationship in graph_document.relationships:
            if relationship.type == relation_name:
                if relationship.source.type == wrong_source_type:
                    source = relationship.source
                    target = relationship.target
                    relationship.source = target
                    relationship.target = source


def prune_dangling_relationships():
    # now remove all dangling relationships
    for graph_document in graph_documents:
        relationships_to_remove = []
        for relationship in graph_document.relationships:
            if is_not_a_listed_product(
                relationship.source
            ) or is_not_a_listed_product(relationship.target):
                relationships_to_remove.append(relationship)
        for relationship in relationships_to_remove:
            graph_document.relationships.remove(relationship)


def prune_unwanted_relationships(relation_name, source, target):
    node_types = {source, target}
    for graph_document in graph_documents:
        relationships_to_remove = []
        for relationship in graph_document.relationships:
            if (
                relationship.type == relation_name
                and {relationship.source.type, relationship.target.type}
                == node_types
            ):
                relationships_to_remove.append(relationship)
        for relationship in relationships_to_remove:
            graph_document.relationships.remove(relationship)


prune_invalid_products()
prune_invalid_segments({"Home", "Office", "Fitness"})
prune_unwanted_relationships("IN_CATEGORY", "Bundle", "Category")
prune_unwanted_relationships("IN_CATEGORY", "Deal", "Category")
prune_unwanted_relationships("IN_SEGMENT", "Bundle", "Segment")
prune_unwanted_relationships("IN_SEGMENT", "Deal", "Segment")
prune_dangling_relationships()
fix_directions("HAS_DEAL", "Deal")
fix_directions("IN_BUNDLE", "Bundle")

print_graph(graph_documents[:10])

### Load data to Spanner Graph
Cleanup database from previous iterations and load extracted Knowledge Graph
!!! THIS COULD REMOVE DATA FROM YOUR DATABASE !!!

In [ ]:
graph_store.cleanup()
graph_store.add_graph_documents(graph_documents)

### Visualization

Visualizes Graph Data: %%spanner_graph extension renders a visual topology of your nodes and edges of the graph.

In [ ]:
%load_ext spanner_graphs

In [ ]:
%%spanner_graph --project {PROJECT_ID} --instance {INSTANCE} --database {DATABASE}

GRAPH retail_demo_graph
MATCH p = ()->()
RETURN TO_JSON(p) AS path_json

## GraphRAG flow using Spanner Graph

Define helper method to format and print content of retrieved chunks:

In [ ]:
def format_docs(docs):
    print("Context Retrieved: \n")
    for doc in docs:
        print("-" * 80)
        pprint.pprint(json.loads(doc.page_content)[0], width=80, indent=4)
        print("-" * 80)
        print("\n")

    context = "\n\n".join(doc.page_content for doc in docs)
    return context

Define prompt template for the LLM:

In [ ]:
SPANNERGRAPH_QA_TEMPLATE = """
You are a helpful and friendly AI assistant for question answering tasks for an electronics
retail online store.
Create a human readable answer for the for the question.
You should only use the information provided in the context and not use your internal knowledge.
Don't add any information.
Here is an example:

Question: Which funds own assets over 10M?
Context:[name:ABC Fund, name:Star fund]"
Helpful Answer: ABC Fund and Star fund have assets over 10M.

Follow this example when generating answers.
You are given the following information:
- `Question`: the natural language question from the user
- `Graph Schema`: contains the schema of the graph database
- `Graph Query`: A Spanner Graph GQL query equivalent of the question from the user used to extract context from the graph database
- `Context`: The response from the graph database as context. The context has nodes and edges. Use the relationships.
Information:
Question: {question}
Graph Schema: {graph_schema}
Context: {context}

Format your answer to be human readable.
Use the relationships in the context to answer the question.
Only include information that is relevant to a customer.
Helpful Answer:"""

prompt = PromptTemplate(
    template=SPANNERGRAPH_QA_TEMPLATE,
    input_variables=["question", "graph_schema", "context"],
)

llm = ChatVertexAI(model=GENERATIVE_MODEL, temperature=0)

chain = prompt | llm | StrOutputParser()

### Specify user query:

In [ ]:
USER_QUERY = "Give me recommendations for a beginner drone with a good battery and camera and options to exntend"

GraphRAG using Vector Search and Graph Expansion

#### Define utility method for graph rag

In [ ]:
def use_node_vector_retriever(
    question, graph_store, embedding_service, label_expr, expand_by_hops
):
    retriever = SpannerGraphVectorContextRetriever.from_params(
        graph_store=graph_store,
        embedding_service=embedding_service,
        label_expr=label_expr,
        expand_by_hops=expand_by_hops,
        top_k=3,
        # k=10,
    )
    context = format_docs(retriever.invoke(question))
    return context

#### Query retriever to get context for grounded answer generation:

In [ ]:
embedding_service = VertexAIEmbeddings(model_name=EMBEDDING_MODEL)

question = USER_QUERY

context = use_node_vector_retriever(
    question,
    graph_store,
    embedding_service,
    label_expr="Product",
    expand_by_hops=1,
)

### Now lets test GraphRAG to explore output results:

In [ ]:
answer = chain.invoke(
    {
        "question": question,
        "graph_schema": graph_store.get_schema,
        "context": context,
    }
)

print("\n\nAnswer:\n")
print(textwrap.fill(answer, width=80))

## Compare with Conventional RAG

In [ ]:
TABLE_NAME = "rag_table"

Setup and load data for vector search

In [ ]:
def load_data_for_vector_search(splits):
    embeddings = VertexAIEmbeddings(model_name=EMBEDDING_MODEL)

    SpannerVectorStore.init_vector_store_table(
        instance_id=INSTANCE,
        database_id=DATABASE,
        table_name=TABLE_NAME,
    )
    db = SpannerVectorStore(
        instance_id=INSTANCE,
        database_id=DATABASE,
        table_name=TABLE_NAME,
        embedding_service=embeddings,
    )
    # Add the chunks to Spanner Vector Store with embeddings
    ids = [str(uuid.uuid4()) for _ in range(len(splits))]
    row_ids = db.add_documents(splits, ids)


# Create splits for documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250, chunk_overlap=100
)
splits = text_splitter.split_documents(
    [document for document_list in document_lists for document in document_list]
)

# Initialize table and load data
embeddings = VertexAIEmbeddings(model_name=EMBEDDING_MODEL)
load_data_for_vector_search(splits)

### Retrieve and generate using the vanilla RAG approach:

In [ ]:
def format_docs(docs):
    print("Context Retrieved: \n")
    for doc in docs:
        print("-" * 80)
        print(textwrap.fill(doc.page_content, width=80))
        print("-" * 80)
        print("\n")

    context = "\n\n".join(doc.page_content for doc in docs)
    return context

prompt = PromptTemplate(
    template="""
    You are a friendly digital shopping assistant.
    Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Question: {question}
    Context: {context}
    Answer:
  """,
    input_variables=["context", "question"],
)

### Define embeddings model and vector database to use it for RAG

In [ ]:
embeddings = VertexAIEmbeddings(model_name=EMBEDDING_MODEL)

db = SpannerVectorStore(
    instance_id=INSTANCE,
    database_id=DATABASE,
    table_name=TABLE_NAME,
    embedding_service=embeddings,
)
vector_retriever = db.as_retriever(search_kwargs={"k": 3})

### Create a complete pipeline for Retrieval-Augmented Generation (RAG)

In [ ]:
rag_chain = (
        {
            "context": vector_retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
)

### Now lets test traditional RAG approach to compare output results:

In [ ]:
resp = rag_chain.invoke(USER_QUERY)
print("\n\nRag Response:\n")
print(textwrap.fill(resp, width=80))

## Clean up the graph

> USE IT WITH CAUTION!

**Uncomment the next line to clean up all the nodes/edges in your graph and remove your graph definition.**

In [ ]:
#graph_store.cleanup()

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.